# Phase 3. 일반화 테스트: 반도체 특화 모델 유효성 검증

## 목표
- 반도체 데이터로 학습한 모델을 빅테크 섹터에 적용
- 섹터간 성능 차이를 통해 반도체 특화 모델의 유효성 증명
- "반도체 섹터만의 공통 패턴이 존재한다"는 가설 최종 검증

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import yfinance as yf
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data_loader import download_all
from src.features   import process_all, build_combined_dataset, add_technical_indicators, add_target, remove_nan
from src.model      import split_data, train_model, evaluate_model, FEATURE_COLS

# 반도체 데이터 로드 (학습용)
stock_data = download_all(save_csv=False)
processed  = process_all(stock_data, save_csv=False)
combined   = build_combined_dataset(processed)

print("반도체 데이터 로드 완료")
print(f"반도체 데이터 크기: {combined.shape}")

## 1. 빅테크 데이터 수집

반도체 모델을 테스트할 새로운 섹터 데이터를 수집한다.
학습에는 사용하지 않고 테스트에만 사용한다.

In [ ]:
BIGTECH_TICKERS = {
    "AAPL" : "Apple",
    "MSFT" : "Microsoft",
    "GOOGL": "Google",
    "META" : "Meta",
    "AMZN" : "Amazon"
}

START_DATE = "2018-01-01"
END_DATE   = "2025-12-31"

def download_bigtech() -> pd.DataFrame:
    """빅테크 종목 데이터를 수집하고 전처리한다."""
    dfs = []
    
    for ticker in BIGTECH_TICKERS:
        print(f"[{ticker}] 다운로드 중...")
        
        df = yf.download(
            tickers     = ticker,
            start       = START_DATE,
            end         = END_DATE,
            auto_adjust = True,
            progress    = False
        )
        
        # 멀티컬럼 제거
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        df["Ticker"] = ticker
        
        # 전처리
        df = add_technical_indicators(df)
        df = add_target(df)
        df = remove_nan(df)
        
        dfs.append(df)
        print(f"[{ticker}] 완료 → {len(df)}행\n")
    
    combined_bt = pd.concat(dfs, axis=0).sort_index()
    print(f"빅테크 통합 데이터: {len(combined_bt)}행")
    return combined_bt

bigtech = download_bigtech()

In [ ]:
## 2. 실험 설계

### 3가지 시나리오 비교

| 시나리오 | 학습 데이터 | 테스트 데이터 | 목적 |
|---------|-----------|-------------|------|
| A | 반도체 | 반도체 | 기준 성능 (Phase 1) |
| B | 반도체 | 빅테크 | 반도체 모델의 범용성 테스트 |
| C | 빅테크 | 빅테크 | 빅테크 전용 모델 성능 |

B vs C 차이가 클수록
→ 반도체 특화 모델이 빅테크에 맞지 않음
→ 섹터 특화 모델의 유효성 증명

In [ ]:
results_all = {}

# ── 시나리오 A: 반도체 → 반도체 ─────────────────────────────
print("=" * 50)
print("시나리오 A: 반도체 학습 → 반도체 테스트")
print("=" * 50)

X_train_semi, X_test_semi, y_train_semi, y_test_semi = split_data(combined)
model_semi   = train_model(X_train_semi, y_train_semi)
results_semi = evaluate_model(model_semi, X_test_semi, y_test_semi)

results_all["A. 반도체→반도체"] = {
    "accuracy": results_semi["accuracy"],
    "auc"     : results_semi["auc"]
}

# ── 시나리오 B: 반도체 학습 → 빅테크 테스트 ──────────────────
print("\n" + "=" * 50)
print("시나리오 B: 반도체 학습 → 빅테크 테스트")
print("=" * 50)

X_bigtech = bigtech[FEATURE_COLS]
y_bigtech = bigtech["Target"]

y_pred_b = model_semi.predict(X_bigtech)
y_prob_b = model_semi.predict_proba(X_bigtech)[:, 1]

acc_b = accuracy_score(y_bigtech, y_pred_b)
auc_b = roc_auc_score(y_bigtech, y_prob_b)

print(f"정확도: {acc_b:.4f}")
print(f"AUC   : {auc_b:.4f}")

results_all["B. 반도체→빅테크"] = {
    "accuracy": acc_b,
    "auc"     : auc_b
}

# ── 시나리오 C: 빅테크 학습 → 빅테크 테스트 ──────────────────
print("\n" + "=" * 50)
print("시나리오 C: 빅테크 학습 → 빅테크 테스트")
print("=" * 50)

X_train_bt, X_test_bt, y_train_bt, y_test_bt = split_data(bigtech)
model_bt   = train_model(X_train_bt, y_train_bt)
results_bt = evaluate_model(model_bt, X_test_bt, y_test_bt)

results_all["C. 빅테크→빅테크"] = {
    "accuracy": results_bt["accuracy"],
    "auc"     : results_bt["auc"]
}

# ── 결과 표 ──────────────────────────────────────────────────
summary = pd.DataFrame(results_all).T
print("\n시나리오별 성능 비교")
display(summary.round(4))

In [ ]:
# 빅테크 종목별 개별 성능 분석
print("빅테크 종목별 성능 (반도체 모델 적용)")
print("=" * 50)

ticker_results = []

for ticker in BIGTECH_TICKERS:
    df_ticker = bigtech[bigtech["Ticker"] == ticker]
    
    X_t = df_ticker[FEATURE_COLS]
    y_t = df_ticker["Target"]
    
    y_prob_t = model_semi.predict_proba(X_t)[:, 1]
    y_pred_t = model_semi.predict(X_t)
    
    acc_t = accuracy_score(y_t, y_pred_t)
    auc_t = roc_auc_score(y_t, y_prob_t)
    
    ticker_results.append({
        "종목"    : ticker,
        "기업"    : BIGTECH_TICKERS[ticker],
        "Accuracy": acc_t,
        "AUC"     : auc_t
    })
    print(f"[{ticker}] Accuracy: {acc_t:.4f} | AUC: {auc_t:.4f}")

ticker_df = pd.DataFrame(ticker_results).set_index("종목")
display(ticker_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── 왼쪽: 시나리오별 AUC 비교 ────────────────────────────────
scenarios = list(results_all.keys())
aucs      = [v["auc"] for v in results_all.values()]
accs      = [v["accuracy"] for v in results_all.values()]
colors    = ["#3498db", "#e74c3c", "#2ecc71"]

bars = axes[0].bar(scenarios, aucs, color=colors,
                   edgecolor="white", width=0.5)
axes[0].axhline(y=0.5, color="black", linestyle="--",
                alpha=0.5, label="랜덤 기준선")
axes[0].set_title("시나리오별 AUC 비교\n(반도체 특화 모델 유효성 검증)",
                  fontsize=13, fontweight="bold")
axes[0].set_ylabel("AUC Score")
axes[0].set_ylim(0.48, 0.60)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

for bar, val in zip(bars, aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f"{val:.4f}",
                 ha="center", fontsize=11, fontweight="bold")

# ── 오른쪽: 빅테크 종목별 AUC ────────────────────────────────
bt_tickers = ticker_df.index.tolist()
bt_aucs    = ticker_df["AUC"].tolist()

# 반도체 평균 AUC 기준선
semi_auc_avg = results_semi["auc"]

bars2 = axes[1].bar(bt_tickers, bt_aucs,
                    color="#e74c3c", edgecolor="white", width=0.5)
axes[1].axhline(y=semi_auc_avg, color="blue", linestyle="--",
                linewidth=2, label=f"반도체 AUC ({semi_auc_avg:.4f})")
axes[1].axhline(y=0.5, color="black", linestyle="--",
                alpha=0.5, label="랜덤 기준선")
axes[1].set_title("빅테크 종목별 AUC\n(반도체 모델 적용 시)",
                  fontsize=13, fontweight="bold")
axes[1].set_ylabel("AUC Score")
axes[1].set_ylim(0.48, 0.60)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

for bar, val in zip(bars2, bt_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f"{val:.4f}",
                 ha="center", fontsize=11, fontweight="bold")

plt.suptitle("일반화 테스트: 반도체 특화 모델 유효성 검증",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/figures/generalization_test.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 3. 결론

### 핵심 질문
> "반도체 모델이 빅테크에도 통할까?"

### 결과 해석
- 시나리오 A (반도체→반도체) : 기준 성능
- 시나리오 B (반도체→빅테크) : A보다 낮으면 → 섹터 특화 유효
- 시나리오 C (빅테크→빅테크) : B보다 높으면 → 전용 모델 필요

### 최종 결론
B < C 이면
→ 반도체 모델을 빅테크에 그대로 쓰면 성능 저하
→ 섹터마다 고유한 패턴이 존재
→ 반도체 특화 모델의 유효성 증명